<a href="https://colab.research.google.com/github/VizDataWorks/CapitalOne_DataAnalysis/blob/main/ecommerce-challenge/notebooks/00_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 – Data Preparation: Cleaning the Raw Files

**Capital One E-Commerce Data Challenge · Step 1 of 6**

This notebook downloads the three raw datasets, cleans each one, and saves the cleaned tables.

Every table follows the same order: **trim spaces → remove duplicates → fix types and values → validate.**

| Table | Raw | Clean | Output file |
| --- | --- | --- | --- |
| Customers (`buyer.csv`) | 52,500 rows | 47,500 rows | `data/clean/customers.csv` |
| Products (11 vendor files) | 1,037 rows | 1,037 rows | `data/clean/products.csv` |
| Sales (`sales.csv`) | 376,004 rows | 348,905 rows | `data/clean/sales.csv` |

**Project folder:** `ecommerce-challenge/` holds `data/` and `notebooks/`.

*   `data/raw/` has the original files
*   `data/clean/` stores the cleaned files
*   ecommerce_challenge/notebooks' holds Data Analysis notebooks


**Contents:** 1. Setup · 2. Raw files · 3. Helpers · 4. Customers · 5. Products · 6. Sales · 7. Validation · 8. Save clean files · 9. Clean field dictionary

# 1.Setup


In [ ]:
import os, sys, glob, shutil, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

# Where to keep the project folder:
# Google Drive (kept between sessions) or Colab's temporary storage (wiped on disconnect)
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ecommerce-challenge"
elif IN_COLAB:
    BASE = "/content/ecommerce-challenge"
else:
    BASE = os.path.abspath("ecommerce-challenge")

# Project folders (created if missing)
RAW       = os.path.join(BASE, "data", "raw")
VENDORS   = os.path.join(RAW, "vendors")
CLEAN     = os.path.join(BASE, "data", "clean")
NOTEBOOKS = os.path.join(BASE, "notebooks")
for d in (RAW, VENDORS, CLEAN, NOTEBOOKS):
    os.makedirs(d, exist_ok=True)

COMPANY_TZ = "America/Chicago"   # company is in Missouri (Central time)

# Show more columns when displaying tables
pd.set_option("display.max_columns", 30, "display.width", 160, "display.max_colwidth", 60)
print("Project folder:", BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/ecommerce-challenge


# 2.Raw Files

In [ ]:
# Download the raw data from the Capital One challenge repo (only if it isn't here yet)
REPO = "https://github.com/CapitalOneRecruiting/DA-E-Commerce-Data-Challenge.git"

if not os.path.exists(os.path.join(RAW, "sales.csv")):
    # Clone the repo into a temporary folder (latest version only, no history)
    tmp = os.path.join(BASE, "_repo_tmp")
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO, tmp], check=True)

    # Copy the files we need into data/raw and data/raw/vendors
    for f in ["buyer.csv", "sales.csv", "metadata.pdf"]:
        shutil.copy(os.path.join(tmp, f), RAW)
    for f in glob.glob(os.path.join(tmp, "Vendor Datasets", "*.csv")):
        shutil.copy(f, VENDORS)

    # Remove the temporary clone
    shutil.rmtree(tmp)


def show_tree(root):
    """Print a folder tree with file sizes (skips hidden and temp folders)."""
    for path, dirs, files in sorted(os.walk(root)):
        dirs[:] = sorted(d for d in dirs if not d.startswith((".", "_")))
        depth = path[len(root):].count(os.sep)
        print("    " * depth + os.path.basename(path) + "/")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(path, f)) / 1e6
            print("    " * (depth + 1) + f"{f}  ({size:.2f} MB)")


# Check the files arrived
show_tree(os.path.join(BASE, "data"))

data/
    clean/
        customers.csv  (6.29 MB)
        products.csv  (0.13 MB)
        sales.csv  (33.82 MB)
    raw/
        buyer.csv  (6.97 MB)
        metadata.pdf  (0.11 MB)
        sales.csv  (25.40 MB)
        vendors/
            CPSFP.csv  (0.01 MB)
            EYMKE.csv  (0.01 MB)
            JDTOJ.csv  (0.01 MB)
            JSCZL.csv  (0.00 MB)
            KWOXE.csv  (0.01 MB)
            NKDLI.csv  (0.00 MB)
            PQYYY.csv  (0.00 MB)
            ROPIK.csv  (0.01 MB)
            SOPAJ.csv  (0.01 MB)
            TAAFL.csv  (0.01 MB)
            WQEPS.csv  (0.02 MB)


## 3. Helper functions

Shared by all three tables. Each cleaning step prints the number of rows it changed. *(These lines could also be collected into a data quality log table if needed.)*

In [ ]:
def trim(df):
    """Strip leading/trailing spaces from every column (all read as text); blank strings become NaN."""
    return df.apply(lambda c: c.str.strip().replace("", np.nan))


def parse_dates(s):
    """Parse the mixed date formats found in the raw files.

    Formats seen: 'm/d/yy H:MM', 'm/d/yy', 'dd-Mon-yy' and Unix seconds (e.g. 1716118560).
    '1/1/70' is a placeholder for "no date" and becomes NaT.
    """
    s = s.astype("string")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # Unix timestamps: 9-10 digits, in seconds
    unix = s.str.fullmatch(r"\d{9,10}(\.0)?", na=False)
    out[unix] = pd.to_datetime(s[unix].astype(float), unit="s")

    # Try each text format in turn, only on values not parsed yet
    for fmt in ["%m/%d/%y %H:%M", "%m/%d/%y", "%d-%b-%y"]:
        todo = ~unix & s.notna() & out.isna()
        out[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")


    # '1/1/70' means "no date"
    out[s.str.startswith("1/1/70", na=False)] = pd.NaT
    return out

    #    """Convert buyer-local times to company time (Central)."""
    def to_company_time(local, tz):
      out = pd.Series(pd.NaT, index=local.index, dtype="datetime64[ns]")
      for zone in tz.dropna().unique():
        m = (tz == zone) & local.notna()
        # Read the time in the buyer's time zone, then convert to company time
        out[m] = (local[m].dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT")
                          .dt.tz_convert(COMPANY_TZ).dt.tz_localize(None))
    return out

## 4. Customers (`buyer.csv`): 52,500 rows → 47,500

*   Spaces trimmed first: 2,500 rows. Example - `" B123"` and
`"B123"` were merged be the same buyer.
*   2,500 exact duplicate rows removed.
*   Duplicate buyer_ids checked: the raw file has 4,999. After the
exact copies are removed, every remaining duplicate has a blank ID, so no real customer records needed merging
*   2,500 rows with a blank buyer_id dropped, because they can never be joined to an order.
*   Signup dates modified into real dates.
*   Blank region (4,763) and timezone (2,369) filled from the state. The notebook first confirms that each state has only one region and one timezone, so the fill is exact.
*  `is_referred` = 2 (473 rows) set to unknown rather than guessed, so it doesn't bias the referral analysis.



In [ ]:
def load_customers(path=os.path.join(RAW, "buyer.csv")):
    df = pd.read_csv(path, dtype=str)

    # 1. Trim spaces
    df = trim(df)

    # 2. Duplicates
    df = df.drop_duplicates()

    # 3. Fix types and values

    # Drop rows with no buyer_id: without an ID, a customer can't be joined to their orders
    df = df[df.buyer_id.notna()].copy()

    # Convert signup_date from text to a real date (time removed)
    df["signup_date"] = parse_dates(df.signup_date).dt.normalize()

    # Fill blank region and timezone using the customer's state.
    for col in ["region", "timezone"]:
        per_state = df.dropna(subset=[col]).groupby("state")[col].nunique()
        assert (per_state == 1).all(), f"a state has more than one {col}"
        lookup = df.dropna(subset=[col]).drop_duplicates("state").set_index("state")[col]
        df[col] = df[col].fillna(df.state.map(lookup))

    # is_referred should only be 0 (no) or 1 (yes).
    df["is_referred"] = pd.to_numeric(df.is_referred).where(lambda x: x.isin([0, 1])).astype("Int64")
    for c in ["is_active_buyer", "wishlist_size"]:
        df[c] = pd.to_numeric(df[c]).astype("Int64")

    # 4. Validate
    assert df.buyer_id.is_unique
    return df.sort_values("buyer_id").reset_index(drop=True)

customers = load_customers()
display(customers.head())

,buyer_id,customer_group,is_active_buyer,signup_date,customer_segment,preferred_subcategory,subcategory_pool,preferred_channel,preferred_payment,region,state,timezone,is_referred,wishlist_size
0,B10000,VIP,1,2018-09-08,Fashion Shopper,Skirts,"Skirts, Jeans",App,Debit Card,West,NV,America/Los_Angeles,0,28
1,B10001,VIP,1,2019-08-20,Tech Enthusiast,Bluetooth Speakers,"Power Banks, Smartphones",Website,Credit Card,West,NV,America/Los_Angeles,0,9
2,B10002,VIP,1,2018-08-22,Fitness Buff,Yoga Mats,"Water Bottles, Yoga Mats",App,Digital Wallet,East,NJ,America/New_York,0,2
3,B10003,VIP,1,2023-04-23,Cozy Homemaker,Table Lamps,"Throw Pillows, Wall Art",Mobile,Digital Wallet,Central,TX,America/Chicago,0,11
4,B10004,VIP,1,2023-10-17,Fashion Shopper,Dresses,"Handbags, Skirts",Website,Digital Wallet,Central,WI,America/Chicago,1,29


## 5. Products (11 vendor files): 1,037 rows
*   Column names made consistent before the union: two files call the margin column `pct_profit_margin`.
*   Margins converted: JDTOJ and ROPIK store margin as a percentage, the other files as a fraction. The median margin per file shows the 100× gap.
*   IDs uppercased.
*   `$` and commas removed from prices (46 rows).
*  True/false flags with mixed spellings made consistent (151 rows).
*   New product key `sku_id`: 245 product numbers are sold by more than one vendor, so the key combines subcategory, vendor and product number.



In [ ]:
# Maps the different spellings of yes/no flags to True/False
FLAG_MAP = {"true": True, "y": True, "1": True, "false": False, "n": False, "0": False}


def load_products(folder=VENDORS):
    """Union the vendor files and clean them. Returns one row per sku_id."""

    # 1. Read every vendor file and stack them into one table
    frames = []
    for f in sorted(glob.glob(os.path.join(folder, "*.csv"))):
        d = pd.read_csv(f, dtype=str)
        # Some files name the margin column differently; rename it so all files match
        d = d.rename(columns={"pct_profit_margin": "profit_margin"})
        # Remember which file each row came from (used to fix margins below)
        d["source_file"] = os.path.basename(f)
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)

    # 2. Trim spaces (blank strings become missing values)
    df = df.apply(lambda s: s.str.strip()).replace("", pd.NA)

    # 3. Profit margin
    # Remove any '%' sign and convert to a number
    df["profit_margin"] = pd.to_numeric(df.profit_margin.str.rstrip("%"))
    # Some files store margin as a percentage (25) and others as a fraction (0.25).
    # A file whose typical (median) margin is above 1 must be using percentages.
    median_by_file = df.groupby("source_file").profit_margin.median()
    pct_files = median_by_file[median_by_file > 1].index
    # Divide those files' margins by 100 so every row is a fraction
    df.loc[df.source_file.isin(pct_files), "profit_margin"] /= 100
    # Note: negative margins are kept on purpose (products sold at a loss)

    # 4. Make ID columns uppercase so the same ID always matches
    id_cols = ["category_id", "subcategory_id", "vendor", "product_num"]
    df[id_cols] = df[id_cols].apply(lambda c: c.str.upper())

    # 5. Price: remove '$' and commas, then convert to a number
    df["price"] = pd.to_numeric(df.price.str.replace(r"[$,]", "", regex=True))

    # 6. Flags: standardise Y / 1 / true / N / 0 / false into True/False
    # "boolean" (lowercase b) is a True/False type that allows missing values
    for c in ["currently_active_vendor", "active_product"]:
        df[c] = df[c].str.lower().map(FLAG_MAP).astype("boolean")

    # Stock as a whole number, keeping blanks as missing (NA)
    df["current_stock"] = pd.to_numeric(df.current_stock).astype("Int64")

    # 7. New product key
    # product_num alone isn't unique (different vendors reuse the same number),
    # so build sku_id from subcategory + vendor + product number
    df.insert(0, "sku_id", df.subcategory_id + "-" + df.vendor + "-" + df.product_num)

    # 8. Validate: every product must have its own sku_id
    assert df.sku_id.is_unique
    return df


products = load_products()
display(products.head())

,sku_id,category_name,category_id,subcategory_name,subcategory_id,vendor,currently_active_vendor,product_num,price,profit_margin,active_product,current_stock,source_file
0,PBESK-CPSFP-9DC69ECE1503,Beauty,PBE,Skincare,PBESK,CPSFP,True,9DC69ECE1503,9.91,0.767131,True,90,CPSFP.csv
1,PBESK-CPSFP-EF523B4FA026,Beauty,PBE,Skincare,PBESK,CPSFP,True,EF523B4FA026,13.42,0.767131,True,65,CPSFP.csv
2,PBESK-CPSFP-C541AC8EF917,Beauty,PBE,Skincare,PBESK,CPSFP,True,C541AC8EF917,60.74,0.767131,True,29,CPSFP.csv
3,PBESK-CPSFP-7CCFBF68193B,Beauty,PBE,Skincare,PBESK,CPSFP,True,7CCFBF68193B,55.82,0.767131,True,91,CPSFP.csv
4,PBESK-CPSFP-97FBE210389E,Beauty,PBE,Skincare,PBESK,CPSFP,True,97FBE210389E,55.17,0.767131,True,17,CPSFP.csv


## 6. Sales (`sales.csv`): 376,004 rows → 348,905


*   Rows removed: 24,878 exact duplicates.
*   2,221 repeated order+product lines collapsed into one. Each keeps its most common quantity, and on a tied status it is marked Cancelled, so revenue is never overstated.
*   Values fixed: 4,429 dates stored as Unix timestamps converted. 179 `1/1/70` placeholder dates set to null.
*   3,545 `Deliverred` typos corrected.
*   Rows kept but flagged:
o	394 bad quantities (missing or outside 1–3) are set to null and flagged.
o	1,662 orders with no status are labelled Unknown and left out of revenue.
o	16,889 lines from buyers missing from the customer table are flagged.
o	159 lines with no buyer_id at all are flagged too.


In [ ]:
def parse_order_dates(s):
    """Convert order dates from text to real datetimes.

    - Unix timestamps (digits only, in seconds) are converted from epoch time
    - '1/1/70' is a placeholder for "no date", so it becomes blank (NaT)
    - Everything else is read as a normal date string
    """
    is_unix = s.str.fullmatch(r"\d+", na=False)
    is_placeholder = s.str.match(r"1/1/(19)?70\b", na=False)
    out = pd.to_datetime(s.where(~is_unix & ~is_placeholder), errors="coerce", format="mixed")
    out[is_unix] = pd.to_datetime(s[is_unix].astype("int64"), unit="s")
    return out


def fill_from_order(df, col):
    """Fill blanks in `col` from other lines of the same order, when that order has exactly one value."""
    vals = df.dropna(subset=[col]).groupby("order_id")[col]
    # Keep only orders where all filled-in lines agree on a single value
    single = vals.first()[vals.nunique() == 1]
    # Fill blanks with that order's value; orders with conflicting values stay blank
    df[col] = df[col].fillna(df.order_id.map(single))
    return df


def collapse_repeated_lines(df, key=("order_id", "sku_id")):
    """Collapse repeated order+product lines into one line.

    quantity     = most common value in the group (tie -> the smaller value)
    order_status = most common value (tie -> 'Cancelled', so revenue is never overstated)
    """
    key = list(key)

    # Separate out the lines whose order+product appears more than once
    dup = df.duplicated(key, keep=False)
    g = df[dup]

    # Quantity: count each value per group and keep the most common.
    # Sorting by count (high first) then quantity (low first) means a tie goes to the smaller value.
    q = (g.dropna(subset=["quantity"])
          .groupby(key + ["quantity"]).size().rename("n").reset_index()
          .sort_values(key + ["n", "quantity"], ascending=[True, True, False, True])
          .drop_duplicates(key))

    # Status: count each status per group and keep only the top one(s)
    s = g.dropna(subset=["order_status"]).groupby(key + ["order_status"]).size().rename("n").reset_index()
    s = s[s.n == s.groupby(key).n.transform("max")]
    s = s.groupby(key).agg(order_status=("order_status", "first"),
                           ties=("order_status", "size")).reset_index()
    # If two or more statuses tie, use 'Cancelled' so revenue is never overstated
    s.loc[s.ties > 1, "order_status"] = "Cancelled"

    # Build one line per group: other columns from the first line, plus the chosen quantity and status
    one = (g.drop_duplicates(key).drop(columns=["quantity", "order_status"])
             .merge(q[key + ["quantity"]], on=key, how="left")
             .merge(s[key + ["order_status"]], on=key, how="left"))

    # Put the collapsed lines back together with the lines that were never repeated
    return pd.concat([df[~dup], one[df.columns]], ignore_index=True)

COMPANY_TZ = "America/Chicago"   # company is in Missouri (Central time)


def to_company_time(local, tz):
    """Convert buyer-local times to company time (Central)."""
    out = pd.Series(pd.NaT, index=local.index, dtype="datetime64[ns]")
    for zone in tz.dropna().unique():
        m = (tz == zone) & local.notna()
        # Read the time in the buyer's time zone, then convert to company time
        out[m] = (local[m].dt.tz_localize(zone, ambiguous="NaT", nonexistent="NaT")
                          .dt.tz_convert(COMPANY_TZ).dt.tz_localize(None))
    return out


def load_sales(path=os.path.join(RAW, "sales.csv"), customers=None):
    """Load and clean the sales lines (one row per order + product)."""
    df = pd.read_csv(path, dtype=str)

    # 1. Trim spaces (blank strings become missing values)
    df = df.apply(lambda s: s.str.strip()).replace("", pd.NA)

    # 2. Remove exact duplicate rows
    df = df.drop_duplicates()

    # 3. Fix values
    # Note which lines include a time of day ('H:MM' or a Unix timestamp).
    # Date-only values like '6/26/24' would otherwise look like midnight.
    has_time = (df.order_datetime.str.contains(":", na=False)
                | df.order_datetime.str.fullmatch(r"\d{9,10}(\.0)?", na=False))
    df["order_datetime"] = parse_order_dates(df.order_datetime)

    # Correct the 'Deliverred' typo
    df["order_status"] = df.order_status.replace({"Deliverred": "Delivered"})

    # Quantity must be a number from 1 to 3; anything else becomes missing (flagged in step 5)
    q = pd.to_numeric(df.quantity, errors="coerce")
    df["quantity"] = q.where(q.between(1, 3)).astype("Int64")

    # Fill blank buyer_id and order_status from other lines of the same order
    df = fill_from_order(df, "buyer_id")
    df = fill_from_order(df, "order_status")
    # Order time in company time: order_datetime is the buyer's local time,
    # so convert it using the buyer's time zone. Lines with no time stay blank.
    buyer_tz = df.buyer_id.map(customers.set_index("buyer_id").timezone)
    df["order_datetime_company"] = to_company_time(df.order_datetime.where(has_time), buyer_tz)
    # Give every line of an order the same time (some orders mix timed and date-only lines)
    df["order_datetime_company"] = df.groupby("order_id").order_datetime_company.transform("min")

    # 4. Collapse repeated order+product lines into one line each
    df = collapse_repeated_lines(df)

    # 5. Flags (rows are kept, just marked)
    df["flag_bad_quantity"] = df.quantity.isna()

    # Orders with no status are labelled 'Unknown'; only 'Delivered' orders count as revenue
    df["order_status"] = df.order_status.fillna("Unknown")
    df["is_revenue"] = df.order_status.eq("Delivered")

    # Flag lines with no buyer, or a buyer that isn't in the customer table
    df["flag_missing_buyer_id"] = df.buyer_id.isna()
    df["flag_buyer_not_in_customers"] = df.buyer_id.notna() & ~df.buyer_id.isin(customers.buyer_id)

    return df.sort_values(["order_datetime", "order_id", "sku_id"]).reset_index(drop=True)


sales = load_sales(customers=customers)
display(sales.head())

,order_id,buyer_id,sku_id,quantity,order_datetime,order_status,order_datetime_company,flag_bad_quantity,is_revenue,flag_missing_buyer_id,flag_buyer_not_in_customers
0,O00000003,B13344,PBEMA-WQEPS-0F47F542BAD6,1,2024-01-01 00:00:00,Delivered,NaT,False,True,False,False
1,O00000024,B15811,PBEMA-CPSFP-0F47F542BAD6,3,2024-01-01 00:00:00,Delivered,NaT,False,True,False,False
2,O00000043,B12027,PCLJA-WQEPS-F7551344E755,1,2024-01-01 00:00:00,Delivered,2024-01-01 16:42:00,False,True,False,False
3,O00000051,B12885,PCLJA-CPSFP-935B98D0317C,1,2024-01-01 00:55:00,Delivered,2024-01-01 02:55:00,False,True,False,False
4,O00000022,B10791,PLESP-SOPAJ-2D31003DCCCA,1,2024-01-01 02:54:00,Delivered,2024-01-01 02:54:00,False,True,False,False


## 7. Validation

These checks must all pass before the clean files are saved. They confirm the keys are unique and that every sales line can join to a product, so later joins cannot add or lose rows.

In [ ]:
checks = {
    "customers: buyer_id unique":                   customers.buyer_id.is_unique,
    "customers: no blank region or timezone":       customers[["region", "timezone"]].notna().all().all(),
    "products: sku_id unique":                      products.sku_id.is_unique,
    "products: margins are fractions (-1 to 1)":    products.profit_margin.between(-1, 1).all(),
    "products: prices numeric and positive":        (products.price > 0).all(),
    "sales: one line per order + sku":              not sales.duplicated(["order_id", "sku_id"]).any(),
    "sales: every sku_id found in products":        sales.sku_id.isin(products.sku_id).all(),
    "sales: quantities are 1-3 or null":            sales.quantity.dropna().between(1, 3).all(),
    "sales: statuses are Delivered/Cancelled/Unknown": set(sales.order_status) <= {"Delivered", "Cancelled", "Unknown"},
    "sales: dates within 2024-2025":                sales.order_datetime.dropna().dt.year.between(2024, 2025).all(),
    "sales: one buyer per order":                   (sales.dropna(subset=["buyer_id"]).groupby("order_id").buyer_id.nunique() == 1).all(),
}
result = pd.DataFrame({"check": list(checks), "passed": [bool(v) for v in checks.values()]})
display(result)
assert result.passed.all(), "a validation check failed"

summary = pd.DataFrame({
    "table": ["customers", "products", "sales"],
    "clean_rows": [len(customers), len(products), len(sales)],
    "columns": [customers.shape[1], products.shape[1], sales.shape[1]],
})
display(summary)
print("Sales status mix:", sales.order_status.value_counts().to_dict())
print("Distinct orders:", f"{sales.order_id.nunique():,}",
      "| date range:", sales.order_datetime.min().date(), "to", sales.order_datetime.max().date())

,check,passed
0,customers: buyer_id unique,True
1,customers: no blank region or timezone,True
2,products: sku_id unique,True
3,products: margins are fractions (-1 to 1),True
4,products: prices numeric and positive,True
5,sales: one line per order + sku,True
6,sales: every sku_id found in products,True
7,sales: quantities are 1-3 or null,True
8,sales: statuses are Delivered/Cancelled/Unknown,True
9,sales: dates within 2024-2025,True


,table,clean_rows,columns
0,customers,47500,14
1,products,1037,13
2,sales,348905,11


Sales status mix: {'Delivered': 336730, 'Cancelled': 10513, 'Unknown': 1662}
Distinct orders: 244,280 | date range: 2024-01-01 to 2025-12-30


## 8. Save the clean files

Written to `data/clean/`. Notebooks 01–05 read these files instead of repeating the cleaning.

In [ ]:
customers.to_csv(os.path.join(CLEAN, "customers.csv"), index=False)
products.to_csv(os.path.join(CLEAN, "products.csv"), index=False)
sales.to_csv(os.path.join(CLEAN, "sales.csv"), index=False)
show_tree(os.path.join(BASE, "data"))

data/
    clean/
        customers.csv  (6.29 MB)
        products.csv  (0.13 MB)
        sales.csv  (40.40 MB)
    raw/
        buyer.csv  (6.97 MB)
        metadata.pdf  (0.11 MB)
        sales.csv  (25.40 MB)
        vendors/
            CPSFP.csv  (0.01 MB)
            EYMKE.csv  (0.01 MB)
            JDTOJ.csv  (0.01 MB)
            JSCZL.csv  (0.00 MB)
            KWOXE.csv  (0.01 MB)
            NKDLI.csv  (0.00 MB)
            PQYYY.csv  (0.00 MB)
            ROPIK.csv  (0.01 MB)
            SOPAJ.csv  (0.01 MB)
            TAAFL.csv  (0.01 MB)
            WQEPS.csv  (0.02 MB)
